In [1]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/rakesh/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [5]:
import cv2
from ultralytics import YOLO

# Load the model
model = YOLO("yolov8n.pt")

# Open the video source (0 for built-in webcam, or 'path/to/video.mp4')
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    
    if success:
        # Run YOLO detection on the current frame
        # stream=True utilizes a generator for better memory management with video
        results = model(frame, stream=True)
        
        for r in results:
            # Get the annotated frame
            annotated_frame = r.plot()
            
            # Display the stream
            cv2.imshow("YOLO Real-Time Detection", annotated_frame)
            
        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        break

# Clean up
cap.release()
cv2.destroyAllWindows()


0: 480x640 3 persons, 1 chair, 16.5ms
Speed: 0.8ms preprocess, 16.5ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 1 chair, 14.4ms
Speed: 0.8ms preprocess, 14.4ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 1 chair, 16.0ms
Speed: 0.5ms preprocess, 16.0ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 1 chair, 13.8ms
Speed: 0.5ms preprocess, 13.8ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 1 chair, 14.3ms
Speed: 0.6ms preprocess, 14.3ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 1 chair, 14.0ms
Speed: 0.6ms preprocess, 14.0ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 persons, 1 chair, 15.7ms
Speed: 0.8ms preprocess, 15.7ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 1 chair, 13.9ms
Speed: 0.

In [4]:
from ultralytics import YOLO

# 1. Load the YOLO detection model (automatically downloads the weights if not present)
# Replace 'yolov8n.pt' or 'yolo11n.pt' with your specific model version (e.g., n, s, m, l, x)
model = YOLO("yolov8n.pt")

# 2. Run inference on an image
# You can use a local path ('path/to/image.jpg') or a web URL
source = "image3.jpg"
results = model(source)

# 3. Process and display the results
for result in results:
    # Print detected bounding boxes, confidence scores, and class IDs to the console
    print("Boxes:", result.boxes.xyxy)      # [xmin, ymin, xmax, ymax]
    print("Confidences:", result.boxes.conf)  # Confidence scores
    print("Class IDs:", result.boxes.cls)      # Class indexes (e.g., 0 for person)
    
    # Show the image with bounding boxes drawn on top
    result.show()
    
    # Save the annotated image to the default directory (runs/detect/predict/)
    result.save(filename="detected_result.jpg")


image 1/1 /home/rakesh/GitRepo/Object-Detection-Models/image3.jpg: 384x640 1 banana, 1 apple, 24.0ms
Speed: 1.9ms preprocess, 24.0ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
Boxes: tensor([[152.8540,  43.6977, 252.2990, 136.3970],
        [ 18.4016,  30.5332, 143.0974, 134.3175]])
Confidences: tensor([0.9080, 0.8760])
Class IDs: tensor([47., 46.])


/usr/bin/eog: symbol lookup error: /snap/core20/current/lib/x86_64-linux-gnu/libpthread.so.0: undefined symbol: __libc_pthread_init, version GLIBC_PRIVATE


#### **Custom Folder Inference Pipeline**

**Key Optimizations in this Pipeline**

- **stream=True**: Crucial for large datasets. Instead of loading all images and their predictions into your system RAM simultaneously, it streams them iteratively.
- **result.path**: Allows you to dynamically grab the file names from your input source so your outputs stay systematically organized.
- **result.plot()**: Generates a standard NumPy array with the drawn bounding boxes if you plan to do further image processing (like cropping or adding custom overlays with OpenCV).

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

# 1. Initialize the model
# Using yolo26n as referenced in the Ultralytics suite
model = YOLO("yolo26n.pt") 

# 2. Define your folder paths
input_folder = "path/to/your/input_images"
output_folder = "path/to/your/saved_results"

# Ensure the output directory exists
os.makedirs(output_folder, exist_ok=True)

# 3. Run inference on the entire folder
# stream=True processes images one by one (generator mode) to prevent memory overload
results = model(input_folder, stream=True)

# 4. Iterate through the results and process each image
for result in results:
    # Get the original filename (e.g., 'photo1.jpg')
    original_path = Path(result.path)
    filename = original_path.name
    
    print(f"Processing: {filename}...")
    
    # --- Example A: Extracting Bounding Boxes to use in your code ---
    boxes = result.boxes.xyxy    # Coordinates: [xmin, ymin, xmax, ymax]
    scores = result.boxes.conf   # Confidence scores
    clss = result.boxes.cls      # Class IDs
    
    for box, score, cls in zip(boxes, scores, clss):
        class_name = model.names[int(cls)]
        print(f"  - Found {class_name} with confidence {score:.2f} at {box.tolist()}")

    # --- Example B: Saving the visual bounding boxes to your custom folder ---
    # Plot drawing onto the image frame
    annotated_frame = result.plot() 
    
    # Save the visual frame directly to your specific output folder
    save_path = os.path.join(output_folder, f"detected_{filename}")
    result.save(filename=save_path)

print(f"\n Pipeline complete! All annotated images saved to: {output_folder}")